### 1.2. Phân tích Khám phá Dữ liệu (Exploratory Data Analysis - EDA)
Trước khi đưa dữ liệu vào mô hình hồi quy, ta tiến hành khảo sát phân phối của biến mục tiêu `Salary` và mối quan hệ tuyến tính giữa các chỉ số kỹ thuật nhằm phát hiện các vấn đề toán học (lệch phân phối, đa cộng tuyến).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Vẽ phân phối biến mục tiêu trước và sau khi lấy Logarit
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(df['Salary'], kde=True, color='#d62728', ax=axes[0])
axes[0].set_title('Phân Phối Lương Thô (Lệch phải nặng)')


sns.histplot(train_final['Log_Salary'], kde=True, color='#1f77b4', ax=axes[1])
axes[1].set_title('Phân Phối Lương Sau Biến Đổi Logarit (Chuẩn hóa)')
plt.show()

# 2. Vẽ ma trận tương quan để kiểm tra hiện tượng đa cộng tuyến (Multicollinearity)
core_feats = ['Age', 'GP', 'MP', 'FG', 'FGA', '3P', 'TRB', 'AST', 'STL', 'BLK', 'PTS', 'SEASON_EXP']
plt.figure(figsize=(11, 8))
sns.heatmap(df[core_feats].corr(), annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Ma Trận Tương Quan Hệ Số Pearson Giữa Các Đặc Trưng Số Cốt Lõi')
plt.tight_layout()
plt.show()

## 4. Giải thích Mô hình & Độ quan trọng đặc trưng (Feature Importance)
Dựa trên mô hình tối ưu nhất thu được từ quá trình thực nghiệm (**OLS Chọn Biến**), ta tiến hành trích xuất trị tuyệt đối của các hệ số hồi quy đã chuẩn hóa để đánh giá xem những nhân tố kỹ thuật nào có sức ảnh hưởng mạnh nhất đến mức lương của một cầu thủ NBA.

In [ ]:

ols_selected_res = results["OLS Chọn Biến"]
features_real = ols_selected_res["features"]
coefs_real = ols_selected_res["coefficients"][1:] # Bỏ Intercept ở vị trí đầu tiên

# Tạo DataFrame xử lý sắp xếp
df_importance = pd.DataFrame({
    'Feature': features_real,
    'Coefficient': coefs_real
})
df_importance['Importance'] = df_importance['Coefficient'].abs()

#Vẽ biểu đồ tầm quan trọng của các đặc trưng (Feature Importance)
df_importance = df_importance.sort_values(by='Importance', ascending=False).head(15).iloc[::-1]
colors = ['#1f77b4' if c >= 0 else '#d62728' for c in df_importance['Coefficient']]

# Thực hiện vẽ
plt.figure(figsize=(12, 7))
bars = plt.barh(df_importance['Feature'], df_importance['Importance'], color=colors, height=0.6)

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.005, bar.get_y() + bar.get_height()/2, f'{width:.3f}', va='center', ha='left', fontsize=9, fontweight='bold')

plt.title('Top Đặc Trưng Quan Trọng Nhất Quyết Định Mức Lương Cầu Thủ NBA\n', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Độ quan trọng (Trị tuyệt đối của Hệ số hồi quy đã chuẩn hóa)')
plt.ylabel('Các đặc trưng kỹ thuật')
plt.grid(axis='x', linestyle='--', alpha=0.5)

from matplotlib.patches import Patch
plt.legend(handles=[Patch(facecolor='#1f77b4', label='Tác động thuận chiều (> 0)'), Patch(facecolor='#d62728', label='Tác động nghịch chiều (< 0)')], loc='lower right')
plt.tight_layout()

import os
os.makedirs('report', exist_ok=True)
plt.savefig('report/feature_importance.png', dpi=300)
plt.show()